#### Gold Layer – Analytics / Business Tables

- gold.daily_revenue_by_category – for dashboards
- gold.customer_360 – for customer analytics

#### Create Gold Schema & Tables

In [0]:
use catalog project;

CREATE TABLE IF NOT EXISTS gold.daily_revenue_by_category (
  order_date    DATE,
  category      STRING,
  total_qty     BIGINT,
  gross_revenue DECIMAL(18,2),
  order_count   BIGINT,
  load_ts       TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS gold.customer_360 (
  customer_id     BIGINT,
  customer_name   STRING,
  email           STRING,
  city            STRING,
  state           STRING,
  country         STRING,
  segment         STRING,
  first_order_dt  DATE,
  last_order_dt   DATE,
  total_orders    BIGINT,
  total_spent     DECIMAL(18,2),
  avg_order_value DECIMAL(18,2),
  load_ts         TIMESTAMP
) USING DELTA;


In [0]:
select * from  gold.customer_360;

customer_id,customer_name,email,city,state,country,segment,first_order_dt,last_order_dt,total_orders,total_spent,avg_order_value,load_ts


#### Populate Gold: Daily Revenue by Category

In [0]:
%python
from pyspark.sql.functions import sum as _sum, countDistinct, current_timestamp,col

orders_clean = spark.table("silver.orders_clean")

daily_cat = (
    orders_clean
    .filter("status = 'PAID'")
    .groupBy("order_date", "category")
    .agg(
        _sum("qty").alias("total_qty"),
        _sum(col("qty") * col("price")).alias("gross_revenue"),
        countDistinct("order_id").alias("order_count")
    )
    .withColumn("load_ts", current_timestamp())
)

daily_cat.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.daily_revenue_by_category")


In [0]:
select * from gold.daily_revenue_by_category;

order_date,category,total_qty,gross_revenue,order_count,load_ts
2025-01-01,Books,3,625.00,2,2025-12-13T07:16:42.049056Z
2025-01-01,Electronics,1,500.00,1,2025-12-13T07:16:42.049056Z
2025-01-02,Fashion,3,2100.00,1,2025-12-13T07:16:42.049056Z
2025-01-01,Toys,2,600.00,1,2025-12-13T07:16:42.049056Z
2025-01-02,Electronics,2,1300.00,2,2025-12-13T07:16:42.049056Z


#### Populate Gold: Customer 360

In [0]:
%python
from pyspark.sql.functions import min as _min, max as _max, countDistinct, round

customers_dim = spark.table("silver.customers_dim")

# Aggregate orders by customer
agg_orders = (
    orders_clean
    .filter("status = 'PAID'")
    .groupBy("customer_id")
    .agg(
        _min("order_date").alias("first_order_dt"),
        _max("order_date").alias("last_order_dt"),
        countDistinct("order_id").alias("total_orders"),
        _sum(col("qty") * col("price")).alias("total_spent")
    )
)

customer_360_df = (
    agg_orders.join(customers_dim, "customer_id", "left")
    .withColumn(
        "avg_order_value",
        round(col("total_spent") / col("total_orders"), 2)
    )
    .select(
        "customer_id",
        "customer_name",
        "email",
        "city",
        "state",
        "country",
        "segment",
        "first_order_dt",
        "last_order_dt",
        "total_orders",
        "total_spent",
        "avg_order_value"
    )
    .withColumn("load_ts", current_timestamp())
)

customer_360_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("project.gold.customer_360")


In [0]:
select * from project.gold.customer_360 order by customer_id;

customer_id,customer_name,email,city,state,country,segment,first_order_dt,last_order_dt,total_orders,total_spent,avg_order_value,load_ts
1,Amit Kumar,amit@example.com,Chennai,TN,India,RET,2025-01-01,2025-01-01,1,500.00,500.00,2025-12-13T07:21:13.660067Z
2,Latha R,latha.r@example.com,Bangalore,KA,India,ENT,2025-01-01,2025-01-02,2,1000.00,500.00,2025-12-13T07:21:13.660067Z
2,Latha R,latha.r@example.com,Bangalore,KA,India,ENT,2025-01-01,2025-01-02,2,1000.00,500.00,2025-12-13T07:21:13.660067Z
3,Rahul Jain,rahul@example.com,Hyderabad,TS,India,RET,2025-01-01,2025-01-01,1,125.00,125.00,2025-12-13T07:21:13.660067Z
4,Meena Iyer,meena@example.com,Mumbai,MH,India,ENT,2025-01-01,2025-01-01,1,600.00,600.00,2025-12-13T07:21:13.660067Z
5,Arjun Dev,arjun@example.com,Delhi,DL,India,RET,2025-01-02,2025-01-02,1,2100.00,2100.00,2025-12-13T07:21:13.660067Z
999,null,null,null,null,null,null,2025-01-02,2025-01-02,1,800.00,800.00,2025-12-13T07:21:13.660067Z
